# Distance Prediction dx/dy Overlay On Center Image

Visualize the trained distance prediction model's `dx, dy` output as arrows on the center camera image.

This is a diagnostic overlay, not a calibrated 3D-to-pixel projection. The arrow encodes the model's planar correction vector using a fixed display scale.

In [ ]:
from pathlib import Path
import importlib
import re
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ws_aic').exists():
    PROJECT_ROOT = Path('/home/whyz/aic_sejong')

DISTANCE_PROJECT = PROJECT_ROOT / 'ws_aic/src/ais/ais_distance_prediction'
sys.path.insert(0, str(DISTANCE_PROJECT))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

import model
import model.position_model
import model.utils
importlib.reload(model.position_model)
importlib.reload(model.utils)
importlib.reload(model)

from model import (
    DEFAULT_DATASET_ROOT,
    expand_samples_by_available_ports,
    load_samples,
    port_id_from_sample,
    split_samples_by_stratified_group_kfold,
)
from model.position_model import build_resnet_position_model

plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', 80)

## Configuration

`PIXELS_PER_MM` only controls display arrow length. Increase it if arrows are hard to see.

In [ ]:
DATASET_ROOT = DEFAULT_DATASET_ROOT
CHECKPOINT_PATH = PROJECT_ROOT / 'ws_aic/model/ais_distance_prediction/vision_offset_resnet50_left_center_right_concat/best.pt'
CAMERAS = ('left', 'center', 'right')
CENTER_CAMERA = 'center'
N_SPLITS = 5
FOLD_INDEX = 0
SEED = 42
SPLIT = 'val'  # 'train' or 'val'
NUM_EXAMPLES = 12
PIXELS_PER_MM = 8.0
DISPLAY_SIZE = (576, 512)  # PIL size: width, height
FLIP_IMAGE_Y = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATASET_ROOT, CHECKPOINT_PATH, device

## Load Checkpoint And Model

In [ ]:
payload = torch.load(CHECKPOINT_PATH, map_location='cpu')
config = payload.get('config', {})
state_dict = payload['model_state_dict']

target_mean = torch.as_tensor(config['target_mean'], dtype=torch.float32)
target_std = torch.as_tensor(config['target_std'], dtype=torch.float32)
image_size = config.get('image_size', (256, 288))
if isinstance(image_size, list):
    image_size = tuple(image_size)
checkpoint_cameras = tuple(config.get('cameras', CAMERAS))

if 'head.0.weight' in state_dict:
    hidden_dim = int(state_dict['head.0.weight'].shape[0])
    num_port_heads = int(config.get('num_port_heads', 1))
else:
    head_indices = {
        int(match.group(1))
        for key in state_dict
        if (match := re.match(r'heads\.(\d+)\.0\.weight$', key)) is not None
    }
    hidden_dim = int(state_dict[f'heads.{min(head_indices)}.0.weight'].shape[0])
    num_port_heads = int(config.get('num_port_heads', max(head_indices) + 1))

model_net = build_resnet_position_model(
    backbone_name=config.get('backbone', 'resnet50'),
    pretrained=False,
    output_dim=len(target_mean),
    hidden_dim=hidden_dim,
    dropout=0.1,
    aggregation=config.get('aggregation', 'mean'),
    num_port_heads=num_port_heads,
    num_views=int(config.get('num_views', len(checkpoint_cameras))),
)
model_net.load_state_dict(state_dict)
model_net.to(device).eval()

{'metrics': payload.get('metrics', {}), 'config': config}

## Prepare Samples

Uses the same stratified group split as training.

In [ ]:
raw_samples = load_samples(DATASET_ROOT)
train_samples, val_samples, _ = split_samples_by_stratified_group_kfold(
    raw_samples,
    n_splits=N_SPLITS,
    fold_index=FOLD_INDEX,
    seed=SEED,
    stratify_keys=('task_type', 'port_type'),
)
samples_by_split = {
    'train': expand_samples_by_available_ports(train_samples),
    'val': expand_samples_by_available_ports(val_samples),
}
samples = samples_by_split[SPLIT]
len(samples), samples[0]['sample_id']

## Prediction Helpers

In [ ]:
to_tensor = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

def load_view_tensor(sample, camera):
    path = DATASET_ROOT / sample['images'][camera]
    with Image.open(path) as image:
        image = image.convert('RGB')
        return to_tensor(image)

def raw_target_mm(sample):
    label = sample.get('_port_label') or sample['label']['plug_tip_to_port']
    return np.array([float(label['x_mm']), float(label['y_mm']), float(label['z_mm'])], dtype=np.float64)

@torch.inference_mode()
def predict_sample_mm(sample):
    images = torch.stack([load_view_tensor(sample, camera) for camera in checkpoint_cameras], dim=0)
    port_tensor = torch.tensor([port_id_from_sample(sample)], dtype=torch.long, device=device)
    pred_norm = model_net(images.unsqueeze(0).to(device), port_tensor).cpu()[0]
    pred_mm = pred_norm * target_std + target_mean
    return pred_mm.numpy().astype(np.float64)

def display_center_image(sample):
    path = DATASET_ROOT / sample['images'][CENTER_CAMERA]
    with Image.open(path) as image:
        return image.convert('RGB').resize(DISPLAY_SIZE)

def vector_to_pixels(dx_mm, dy_mm):
    x = float(dx_mm) * PIXELS_PER_MM
    y = float(dy_mm) * PIXELS_PER_MM
    if FLIP_IMAGE_Y:
        y = -y
    return x, y

def draw_arrow(ax, origin, vector_mm, color, label):
    vx, vy = vector_to_pixels(vector_mm[0], vector_mm[1])
    ax.arrow(
        origin[0], origin[1], vx, vy,
        color=color,
        width=2.5,
        head_width=18,
        head_length=18,
        length_includes_head=True,
        alpha=0.9,
    )
    ax.text(origin[0] + vx, origin[1] + vy, label, color=color, fontsize=10, weight='bold')

## Build Prediction Table

In [ ]:
rng = np.random.default_rng(SEED)
indices = rng.choice(len(samples), size=min(NUM_EXAMPLES, len(samples)), replace=False)

records = []
for index in indices:
    sample = samples[int(index)]
    pred = predict_sample_mm(sample)
    target = raw_target_mm(sample)
    error = pred - target
    records.append({
        'index': int(index),
        'sample_id': sample['sample_id'],
        'episode_name': sample['episode_name'],
        'task_type': sample.get('task_type'),
        'port_type': sample.get('port_type'),
        'port_id': port_id_from_sample(sample),
        'pred_dx_mm': pred[0],
        'pred_dy_mm': pred[1],
        'pred_dz_mm': pred[2],
        'target_dx_mm': target[0],
        'target_dy_mm': target[1],
        'target_dz_mm': target[2],
        'xy_error_mm': float(np.linalg.norm(error[:2])),
        'xyz_error_mm': float(np.linalg.norm(error)),
    })

pred_df = pd.DataFrame(records)
pred_df.sort_values('xy_error_mm', ascending=False).head(NUM_EXAMPLES)

## Overlay Predicted And Target dx/dy

Blue arrow: model prediction. Green arrow: target label. Both arrows start from the image center.

In [ ]:
def plot_overlay(row):
    sample = samples[int(row['index'])]
    image = display_center_image(sample)
    width, height = image.size
    origin = (width / 2.0, height / 2.0)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(image)
    ax.scatter([origin[0]], [origin[1]], c='white', s=30, edgecolors='black')
    draw_arrow(ax, origin, (row['pred_dx_mm'], row['pred_dy_mm']), 'dodgerblue', 'pred')
    draw_arrow(ax, origin, (row['target_dx_mm'], row['target_dy_mm']), 'limegreen', 'target')
    ax.set_title(
        f"{row['task_type']} {row['port_type']} port_id={row['port_id']}\n"
        f"pred=({row['pred_dx_mm']:+.1f}, {row['pred_dy_mm']:+.1f})mm  "
        f"target=({row['target_dx_mm']:+.1f}, {row['target_dy_mm']:+.1f})mm  "
        f"xy_err={row['xy_error_mm']:.1f}mm"
    )
    ax.axis('off')
    plt.tight_layout()

for _, row in pred_df.sort_values('xy_error_mm', ascending=False).iterrows():
    plot_overlay(row)

## Error Summary

In [ ]:
pred_df[['xy_error_mm', 'xyz_error_mm']].describe()